### Loading and Preparing Page Index Client

In [ ]:
import os,json,time
from dotenv import load_dotenv
load_dotenv()

page_index_key=os.getenv("PAGE_INDEX_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")


In [ ]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key=page_index_key)
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

### Uploading Pdf to Page Index

In [ ]:
pdf_path = "./LLM_and_RAG_Notes.pdf"
print(f"Uploading {pdf_path}")
result = pi_client.submit_document(pdf_path)
doc_id = result["doc_id"]
print("Uploaded")
print(f"Doc id {doc_id}")

### Building Tree Index

In [21]:
print("Building Tree Index...")
while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"Status: {status}")
    if status=="completed":
        print("Tree Index Is Ready")
        break
    elif status=="failed":
        print("Processing Failed.Check your PDF")
        break
    time.sleep(5)


Building Tree Index...
Status: completed
Tree Index Is Ready


### Inspecting Tree Structure

In [ ]:
tree_result = pi_client.get_tree(doc_id=doc_id,node_summary=True)
pageindex_tree = tree_result.get("result",[])

print(f"Top level section :{len(pageindex_tree)}")
print("Raw tree (first node): ")
print(json.dumps(pageindex_tree[0] if pageindex_tree else [], indent=2))

### Pretty-print the full tree

In [22]:
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("Full Document Structure:\n")
print_tree(pageindex_tree)

Full Document Structure:

[0000] Large Language Models  (p.1)
  └─ [0001] Contents  (p.2)
  └─ [0002] PART 1  (p.4)
  └─ [0003] How LLMs Work  (p.4)
    └─ [0004] 1. What is an LLM?  (p.4)
    └─ [0005] 2. The Input Stage & Tokenization  (p.4)
    └─ [0006] 3. Embeddings (Input & Vector Embeddings)  (p.5)
    └─ [0007] 4. Attention Mechanisms (Single-Head & Multi-Head Attention)  (p.5)
    └─ [0008] 5. Linear Layers & Transformation  (p.6)
    └─ [0009] 6. Softmax & Temperature (Decoding Parameters)  (p.6)
    └─ [0010] 7. Context Windows & Decoders  (p.7)
    └─ [0011] 8. Function / Tool Calling  (p.7)
    └─ [0012] 9. Structured Outputs  (p.8)
  └─ [0013] Foundational RAG Concepts  (p.10)
    └─ [0014] 1. Traditional RAG vs. Standard LLM Interaction  (p.10)
    └─ [0015] 2. Ingestion & Document Architecture  (p.11)
    └─ [0016] 3. Mathematical Representations & Transformers  (p.11)
    └─ [0017] 4. Vector Storage & Retrieval Mechanics  (p.12)
    └─ [0018] 5. Inference, Search & Gen

### Count total nodes

In [ ]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")
print("Each node = one retrievable section of the document")

### LLM Tree Search Function

In [ ]:
def llm_tree_search(query: str, tree: list, model: str = "llama-3.3-70b-versatile") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.

    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """

    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree: {json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

### Test with a sample query

In [ ]:
query = "What are the llm workflows?"

print(f"Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("Selected Node IDs:", result.get("node_list", []))

#### Full End-to-End RAG Pipeline
3 steps:

Tree Search → LLM picks relevant node_ids

Retrieve → Fetch the actual section content from those nodes

Generate → LLM writes a grounded answer with page citations

In [17]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

### Generate answer from retrieved nodes

In [18]:
def generate_answer(query: str, nodes: list, model: str = "llama-3.3-70b-versatile") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "No relevant sections found in the document."

    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


### The complete Vectorless RAG function

In [19]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:

    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")

    return answer

### Run the full pipeline

In [20]:
answer = vectorless_rag(
    query="What are the worflows in llm?",
    tree=pageindex_tree
)

🔍 Query: What are the worflows in llm?

🧠 Reasoning: To find the node IDs most likely containing the answer to the query 'What are the workflows in LLM?', we should look for sections related to Large Language Models (LLM) workflows, LLM applications, or...
🎯 Retrieved node IDs: ['0031', '0027', '0028', '0029', '0030', '0032', '0033', '0034']
📄 Sections found: ['LangChain, LangGraph, MCP & Agent Safety', '1. Tool Calling in LangChain', '2. LangChain vs. LangGraph', '3. LangGraph Execution & State Mechanics', '4. Architectural Patterns for LLM Workflows', '5. Model Context Protocol (MCP)', '6. Guardrails & Safety Mechanisms', '7. Prompt Injection Security']

📝 Answer:
The workflows in LLM are (Section: '4. Architectural Patterns for LLM Workflows' | Page 24): 
1. Prompt Chaining, 
2. Routing, 
3. Parallelization, 
4. Orchestrator-Worker, 
5. Evaluator-Optimizer.
